---
title: "16. End-to-end integration"
description: "Run the local Compose golden path and the separate Azure smoke adapter against the same behavioral definition of done."
---

## Outcome

Chapters 02–07 built the platform one stage at a time on the complete Compose
sandbox: tracked data, reproducible training, batch scoring into the results DB,
online serving behind an exact-version contract, dashboards, and an LLM riding
the same machinery. This chapter proves that the stages compose into one path.

The local instrument is demo/golden_path.py. The cloud instrument is
deploy/smoke-tests.sh or deploy/smoke-tests.ps1. They intentionally use
different trigger mechanisms, because a local runner and the ACA control plane
are different adapters. They preserve the same behavioral definition of done:
terminal job success, results state, exact model identity, readiness, and a
successful prediction.

## The full golden path

~~~mermaid
flowchart TD
    DATA["tracked dataset"]
    TRAIN["train job"]
    VER["MLflow registered version"]
    PROMOTE["production alias + repin"]
    SERVE["serving /readyz + prediction"]
    BATCH["batch parent/child results"]
    CHECK["behavioral assertions"]

    DATA --> TRAIN --> VER --> PROMOTE
    PROMOTE --> SERVE --> CHECK
    PROMOTE --> BATCH --> CHECK
~~~

An LLM app enters at the registered-version step exactly like any other model:
the shared pyfunc loader and results contract keep downstream consumers
indifferent to whether the artifact is tabular or text.


## Two adapters, one evidence set

The local golden path:

1. Triggers training through the local runner and polls its results row.
2. Resolves a newly registered version and promotes it through demo/promote.py.
3. Polls serving /readyz until it reports the exact model name and version.
4. Calls /v1/predictions and verifies the response echoes that version.
5. Triggers batch scoring pinned to the version and polls the parent row to SUCCESS.

The Azure smoke adapter uses az containerapp job start and polls ACA executions
to terminal status. It then checks the dashboard/results API for a successful
batch parent and calls the same serving readiness and prediction routes. The
control-plane calls differ; the assertions do not.

A phase is done when its evidence is demonstrated, not when its chapter has been
read. Local acceptance is demo/golden_path.py finishing with GOLDEN PATH: PASS.
Cloud acceptance is deploy/smoke-tests.sh or .ps1 finishing with all smoke tests
passed.


## Acceptance evidence

| Phase | Evidence on Compose | Evidence on Azure |
|---|---|---|
| Foundation | check_env_contract.py exits 0; docker compose config parses | Terraform footprint and least-privilege identities |
| Training | Registered version with dataset and code lineage | ACA Job execution reaches Succeeded and the results record is honest |
| Batch | Parent/child rows; transient items retry to terminal | ACA execution reaches Succeeded and the dashboard exposes a successful parent |
| Serving | /readyz reports exact version; prediction echoes it; rollback repromotes a known-good version | Same endpoints and assertions through the ACA ingress |
| Observability | Dashboard lists runs and deep-links MLflow | Grafana/alerts and dashboard operate with managed identity |
| LLM | pyfunc registration/evaluation use the Compose profile and bundled fixture | Same train image and entrypoints, with Key Vault credentials and configured dataset |

The local and cloud scripts are separate because their trigger and authentication
mechanisms are different. Their shared definition of done is the contract that
makes the local-first workflow useful: implement and understand the feature on
Compose, then deploy the same workload to ACA and verify the behavior.
